# Fusion raw: séries brutes avec option fenêtres glissantes


In [ ]:
import os
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from pipeline.extract import add_target, load_features_for_approach, load_per_minute_targets
from pipeline.pretraitement import (
    apply_column_aggregations,
    apply_preprocess,
    apply_target_discretization,
    display_target_info,
    prepare_splits_and_impute,
)
from pipeline.raw_segmentation import (
    prepare_splits_and_impute_3d,
    segment_sequences_sliding_window,
)
from pipeline.visu_pretraitement import (
    plot_feature_report,
    plot_preprocessing_report_by_approach,
    plot_split_report,
)
from pipeline.evaluation import (
    evaluate_by_subject,
    evaluate_robustness,
    evaluate_test_set,
    plot_feature_importance,
    plot_pca_if_classification,
    train_with_optional_hyperparameter_search,
)
from pipeline.reporting import export_visual_report

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

print(f"Seed globale: {SEED}")

## 1) Profils modulaires

In [ ]:
DATA_PROFILE = {
    "source": "csv",
    "file_path": r"../data/Données_brutes/DataPhase1_time.csv",
    "subject_id_col": "Participant",
    "time_col": "Time",
}

# ── Ordre des features : IMPORTANT pour le modele multistream ─────────────────
_EYE_FEATURES  = [
    "X gaze direction", "Y gaze direction", "Pupil diameter AVG",
]
_HEAD_FEATURES = [
    "HMDPosX", "HMDPosY", "HMDPosZ",
    "RotX", "RotY", "RotZ",
    "isBoat", "X World Position", "Y World Position",
]
_ALL_FEATURES = _EYE_FEATURES + _HEAD_FEATURES

PREPROCESS_PROFILE = {
    "approach": "B",
    "use_frequency_resampling": True,
    "frequency_resampling_method": "uniform_time_step",
    "uniform_time_step_s": 0.1,

    "apply_temporal_lowpass": False,
    "apply_temporal_highpass": False,
    "apply_temporal_bandpass": True,
    "bandpass_ranges_hz": [[0.08, 0.12], [0.38, 0.42]],
    "bandpass_order": 4,
    "bandpass_min_points": 16,
    "bandpass_features": _ALL_FEATURES,

    "clip_quantiles": [0.05, 0.95],
    "imputation_strategy": "median",
    "drop_low_information_features": True,
    "min_valid_features": 1,
    "normalization": "standard",

    "time_col": "time",
    "time_unit": "s",
    "subject_id_col": "subject_id",
    "exclude_features": ["minute", "time", "sampling_hz", "row_id",
                         "Left Pupil Diameter", "Right Pupil Diameter"],
    "column_aggregations": {
        "Pupil diameter AVG": {
            "columns": ["Left Pupil Diameter", "Right Pupil Diameter"],
            "strategy": "mean",
        },
    },
    "include_features": _ALL_FEATURES,
}

# ── Discrétisation FMS selon Islam et al. (ISMAR 2021) ───────────────────────
#
# Islam et al. utilisent les QUARTILES de la distribution FMS pour définir
# 4 classes : None, Low, Medium, High.
#
#   CS_t = None   si FMS_t <= Q1
#   CS_t = Low    si Q1 < FMS_t <= Q2
#   CS_t = Medium si Q2 < FMS_t <= Q3
#   CS_t = High   si FMS_t > Q3
#
# Sur nos données Phase 1 (564 valeurs FMS, echelle 1-20) :
#   Q1=3  Q2=5  Q3=7
#   -> None  [1-3] : 32%
#   -> Low   (3-5] : 30%
#   -> Medium(5-7] : 16%
#   -> High  (7-20]: 22%
#
#
_Q1, _Q2, _Q3 = 3, 5, 7   # quartiles Phase 1

TARGET_PROFILE = {
    "source": "xlsx",
    "xlsx_path": r"../data/Questionnaires/FMS1_org.xlsx",
    "sheet_name": "Feuil1",
    "subject_id_col": "Sujet",
    "target_mode": "per_minute",
    "target_minute": 14,
    "minute_col": "minute",
    "minute_columns": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],
    "clip_quantiles": None,
    # 4 classes basées sur les quartiles — méthode Islam et al. 2021
    "discretize": {
        "bins":   [0, _Q1, _Q2, _Q3, 20],
        "labels": ["none", "low", "medium", "high"],
    },
}

MODEL_PROFILE = {
    # "cnn_1d"        : Conv1D + GlobalAvgPool
    # "cnn_lstm"      : Conv1D + LSTM (recommande)
    # "bilstm"        : BiLSTM empile
    # "inception_time": multi-echelles state-of-the-art TSC
    # "td_cnn_lstm"   : Time-Distributed CNN-LSTM (Islam et al. ISMAR 2021)
    # "multistream"   : deux branches eye + head (Islam et al. ISMAR 2021)
    "model_type": "multistream",
    "n_eye_features": len(_EYE_FEATURES),
    "task_type": "classification",
    "use_hyperparam_search": False,
    "split_method": "group",
    "test_size": 0.20,
    "val_size": 0.20,
    "class_weight": "balanced",
    "random_state": SEED,
}

RAW_PROFILE = {
    "segmentation_mode": "sliding_window",
    "window_s": 30.0,
    "stride_s": 15.0,
    "sequence_length": 300,
    "pad_value": 0.0,
    "per_subject_norm": True,
}

OUTPUT_PROFILE = {
    "output_dir": r"../data/outputs/fusion_raw",
    "save_visual_report": True,
    "visual_report_format": "both",
    "visual_report_name": "visual_report_fusion_raw",
    "visual_report_functions": [
        "visual_hypothesis_page", "visual_cover_page",
        "visual_model_architecture_page", "visual_split_report",
        "visual_metrics_table_page", "visual_confusion_matrix_page",
        "visual_temporal_preprocess_pages", "visual_missing_values_bar",
        "visual_correlation_pages", "visual_violin_pages",
    ],
    "max_corr_features": 13,
    "max_violin_features": 13,
    "violin_features_per_page": 6,
    "hypothesis": (
        f"Islam et al. 2021 — 4 classes quartiles (Q1={_Q1}, Q2={_Q2}, Q3={_Q3}) "
        f"mode={RAW_PROFILE['segmentation_mode']} "
        f"fenetre={RAW_PROFILE['window_s']}s stride={RAW_PROFILE['stride_s']}s "
        f"modele={MODEL_PROFILE['model_type']}"
    ),
}

_seg = RAW_PROFILE["segmentation_mode"]
_mt  = MODEL_PROFILE["model_type"]
print("Profils charges")
print("Modele              :", _mt)
if _mt == "multistream":
    print(f"  Branche eye  ({len(_EYE_FEATURES)} feat.)")
    print(f"  Branche head ({len(_HEAD_FEATURES)} feat.)")
print("Split               :", MODEL_PROFILE["split_method"])
print("Classes (Islam et al. quartiles) :")
print(f"  none   : FMS <= {_Q1}    (Q1)")
print(f"  low    : {_Q1} < FMS <= {_Q2}  (Q2)")
print(f"  medium : {_Q2} < FMS <= {_Q3}  (Q3)")
print(f"  high   : FMS >  {_Q3}")
if _seg == "sliding_window":
    _w, _s = RAW_PROFILE["window_s"], RAW_PROFILE["stride_s"]
    _ovl   = round(100*(1-_s/_w))
    print(f"Fenetre : {_w}s  stride={_s}s  overlap={_ovl}%")

## 3) Construction de représentation (Approche B)

In [ ]:
features_df = load_features_for_approach(
    data_profile=DATA_PROFILE,
    preprocess_profile=PREPROCESS_PROFILE,
    verbose=True,
)
# Agrégation pupilles (Pupil diameter AVG) avant toute segmentation
features_df = apply_column_aggregations(features_df, PREPROCESS_PROFILE)

print("Shape:", features_df.shape)
display(features_df.head())

## 3.bis) Normalisation per-sujet (optionnelle)

Controlée par `RAW_PROFILE["per_subject_norm"]`.

Centre et réduit chaque feature **par sujet** sur l'ensemble de sa session,
avant toute segmentation ou split. Supprime les biais morphologiques
inter-sujets (taille du HMD → HMDPos, amplitude naturelle du regard, etc.)
sans toucher aux variations intra-sujet qui portent l'information clinique.

In [ ]:
if RAW_PROFILE.get("per_subject_norm", False):
    # Colonnes à normaliser = celles déclarées dans include_features
    _norm_cols = [
        c for c in PREPROCESS_PROFILE["include_features"]
        if c in features_df.columns and pd.api.types.is_numeric_dtype(features_df[c])
    ]
    # Pupil diameter AVG a été créé par apply_column_aggregations → l'inclure si présent
    if "Pupil diameter AVG" in features_df.columns and "Pupil diameter AVG" not in _norm_cols:
        _norm_cols.append("Pupil diameter AVG")

    for sid, idx in features_df.groupby("subject_id", sort=False).groups.items():
        sub = features_df.loc[idx, _norm_cols].astype(float)
        mu    = sub.mean()
        sigma = sub.std().replace(0, 1e-8)
        features_df.loc[idx, _norm_cols] = (sub - mu) / sigma

    print(f"Normalisation per-sujet appliquée")
    print(f"  Sujets    : {features_df['subject_id'].nunique()}")
    print(f"  Features  : {len(_norm_cols)}")
    print(f"  Vérification (mean/std sur un sujet) :")
    _sid0 = features_df["subject_id"].iloc[0]
    _sub0 = features_df.loc[features_df["subject_id"] == _sid0, _norm_cols]
    print(f"    mean ≈ {_sub0.mean().abs().max():.2e}  (attendu ≈ 0)")
    print(f"    std  ≈ {_sub0.std().mean():.4f}       (attendu ≈ 1)")
else:
    print("Normalisation per-sujet : désactivée (RAW_PROFILE['per_subject_norm'] = False)")

## 4) Intégration de la cible

In [ ]:
_seg = RAW_PROFILE["segmentation_mode"]

if _seg == "none":
    # Mode collègue : merge timestep-level, même label pour tous les pas de temps du sujet
    dataset_df = add_target(features_df, TARGET_PROFILE)
    print("Dataset avec cible (mode none):", dataset_df.shape)
    display(dataset_df.head())

else:
    # Mode sliding_window : les cibles sont chargées séparément (per_minute)
    # et seront jointes lors de la segmentation
    target_df = load_per_minute_targets(TARGET_PROFILE)
    target_df["subject_id"] = target_df["subject_id"].astype(str).str.strip()
    target_df = apply_target_discretization(target_df, TARGET_PROFILE)
    print("Cibles par minute:", target_df.shape)
    print("Distribution:")
    print(target_df["target"].value_counts().sort_index())
    display(target_df.head())

## 4.bis) Segmentation en fenêtres glissantes

Actif uniquement si `RAW_PROFILE["segmentation_mode"] == "sliding_window"`.

Chaque fenêtre de `window_s` secondes reçoit le label FMS de la minute
où tombe son centre. La normalisation per-sujet est ensuite appliquée
pour supprimer les différences morphologiques inter-sujets.

In [ ]:
import collections

_seg = RAW_PROFILE["segmentation_mode"]

if _seg == "sliding_window":
    # Prétraitement (filtre + clip) sur la série brute avant segmentation
    proc_df, feature_cols = apply_preprocess(features_df.copy(), PREPROCESS_PROFILE)

    window_s = RAW_PROFILE["window_s"]
    stride_s = RAW_PROFILE["stride_s"]
    T        = RAW_PROFILE["sequence_length"]

    print(f"Segmentation — fenêtre {window_s}s, stride {stride_s}s, T={T}...")
    X_raw, y_raw, groups_raw = segment_sequences_sliding_window(
        raw_df=proc_df,
        feature_cols=feature_cols,
        target_df=target_df,
        window_s=window_s,
        stride_s=stride_s,
        T=T,
        subject_col="subject_id",
        time_col="time",
        pad_value=RAW_PROFILE["pad_value"],
    )

    print(f"X shape : {X_raw.shape}  (n_fenêtres, T, n_features)")
    print(f"Sujets  : {len(set(groups_raw))}")
    print("Distribution des classes :")
    for k, v in sorted(collections.Counter(y_raw).items()):
        print(f"  {k}: {v} ({100*v/len(y_raw):.1f}%)")

    # dataset_df 2D pour les visualisations (moyenne sur T)
    dataset_df = pd.DataFrame(X_raw.mean(axis=1), columns=feature_cols)
    dataset_df["target"]     = y_raw
    dataset_df["subject_id"] = groups_raw
    dataset_df["row_id"]     = np.arange(len(y_raw))

else:
    print("Mode 'none' : pas de segmentation en fenêtres (skip).")
    feature_cols = None  # sera défini à l'étape 5

## 5) Prétraitement, split et préparation

In [ ]:
_seg      = RAW_PROFILE["segmentation_mode"]
task_type = str(MODEL_PROFILE.get("task_type", "classification")).lower()
deep_models = {"cnn_1d", "inception_time", "bilstm", "cnn_lstm", "td_cnn_lstm"}

if _seg == "none":
    # ── Mode collègue : pipeline identique à CNN_1D.ipynb ─────────────────────
    if task_type == "regression" and not pd.api.types.is_numeric_dtype(dataset_df["target"]):
        dataset_df = add_target(features_df, TARGET_PROFILE)
    if task_type == "classification":
        dataset_df = apply_target_discretization(dataset_df, TARGET_PROFILE)

    display_target_info(dataset_df, MODEL_PROFILE, TARGET_PROFILE, PREPROCESS_PROFILE)
    raw_df = dataset_df.copy()
    dataset_df, feature_cols = apply_preprocess(dataset_df, PREPROCESS_PROFILE)

    prepared = prepare_splits_and_impute(
        dataset_df=dataset_df,
        feature_cols=feature_cols,
        preprocess_profile=PREPROCESS_PROFILE,
        model_profile=MODEL_PROFILE,
    )
    X_train_imp = prepared["X_train_imp"]
    X_val_imp   = prepared["X_val_imp"]
    X_test_imp  = prepared["X_test_imp"]
    y_train     = prepared["y_train"]
    y_val       = prepared["y_val"]
    y_test      = prepared["y_test"]
    train_idx, val_idx, test_idx = prepared["train_idx"], prepared["val_idx"], prepared["test_idx"]
    imputer, scaler = prepared["imputer"], prepared["scaler"]

    if task_type == "regression":
        for arr_name in ["y_train", "y_val", "y_test"]:
            locals()[arr_name] = pd.to_numeric(pd.Series(locals()[arr_name]), errors="coerce").to_numpy(dtype=float)

    if MODEL_PROFILE.get("model_type") in deep_models:
        sequence_length = X_train_imp.shape[1]
        n_features_dim  = 1
        X_train_imp = X_train_imp.reshape((-1, sequence_length, n_features_dim))
        X_val_imp   = X_val_imp.reshape((-1,   sequence_length, n_features_dim))
        X_test_imp  = X_test_imp.reshape((-1,  sequence_length, n_features_dim))
        MODEL_PROFILE["n_classes"]       = int(pd.Series(y_train).nunique()) if task_type == "classification" else 1
        MODEL_PROFILE["sequence_length"] = int(sequence_length)
        MODEL_PROFILE["n_features"]      = int(n_features_dim)

else:
    # ── Mode sliding_window : données déjà en 3D (n_fenêtres, T, n_features) ──
    raw_df = dataset_df.copy()

    MODEL_PROFILE["n_classes"]       = int(len(np.unique(y_raw)))
    MODEL_PROFILE["sequence_length"] = int(RAW_PROFILE["sequence_length"])
    MODEL_PROFILE["n_features"]      = int(len(feature_cols))

    prepared = prepare_splits_and_impute_3d(
        X=X_raw,
        y=y_raw,
        groups=groups_raw,
        preprocess_profile=PREPROCESS_PROFILE,
        model_profile=MODEL_PROFILE,
    )
    X_train_imp = prepared["X_train_imp"]
    X_val_imp   = prepared["X_val_imp"]
    X_test_imp  = prepared["X_test_imp"]
    y_train     = prepared["y_train"]
    y_val       = prepared["y_val"]
    y_test      = prepared["y_test"]
    train_idx, val_idx, test_idx = prepared["train_idx"], prepared["val_idx"], prepared["test_idx"]
    imputer, scaler = prepared["imputer"], prepared["scaler"]
    sequence_length = RAW_PROFILE["sequence_length"]
    n_features_dim  = len(feature_cols)

    train_subj = set(groups_raw[train_idx])
    test_subj  = set(groups_raw[test_idx])
    assert len(train_subj & test_subj) == 0, "LEAKAGE : sujets communs train/test !"
    print(f"Split group : {len(train_subj)} sujets train, {len(test_subj)} sujets test — OK ✓")

print(f"\nFeatures ({len(feature_cols)}) : {feature_cols}")
print(f"Split train/val/test : {len(train_idx)} / {len(val_idx)} / {len(test_idx)}")
if MODEL_PROFILE.get("model_type") in deep_models:
    print(f"Shape X_train : {X_train_imp.shape}")
if scaler is not None:
    print("Normalisation :", type(scaler).__name__)

## 5.bis) Visualisation du prétraitement

In [ ]:
plot_preprocessing_report_by_approach(
    raw_df=raw_df,
    processed_df=dataset_df,
    feature_cols=feature_cols,
    preprocess_profile=PREPROCESS_PROFILE,
)
plot_split_report(dataset_df, train_idx, val_idx, test_idx, MODEL_PROFILE)

## 5.ter) Visualisation des features

In [ ]:
plot_feature_report(dataset_df, feature_cols, MODEL_PROFILE, target_profile=TARGET_PROFILE)

## 6) Optimisation hyperparamètres et entraînement

In [ ]:
deep_models = {"cnn_1d", "inception_time", "bilstm", "cnn_lstm", "td_cnn_lstm", "multistream"}
use_search  = bool(MODEL_PROFILE.get("use_hyperparam_search", False))
seq_len     = X_train_imp.shape[1] if MODEL_PROFILE.get("model_type") in deep_models else None
n_feat      = X_train_imp.shape[2] if (MODEL_PROFILE.get("model_type") in deep_models and X_train_imp.ndim == 3) else None

# Pour le modele multistream, injecter n_eye_features dans les params passes au builder
_extra_params = {}
if MODEL_PROFILE.get("model_type") == "multistream":
    _extra_params["n_eye_features"] = MODEL_PROFILE.get("n_eye_features", len(_EYE_FEATURES))
    print(f"Multistream : branche eye={_extra_params['n_eye_features']} features, "
          f"branche head={n_feat - _extra_params['n_eye_features']} features")

# Injecter les params supplementaires dans MODEL_PROFILE pour que build_model les voie
_mp_with_extra = {**MODEL_PROFILE, **_extra_params}

final_model, best_params, results_df = train_with_optional_hyperparameter_search(
    X_train_imp=X_train_imp,
    y_train=y_train,
    X_val_imp=X_val_imp,
    y_val=y_val,
    model_profile=_mp_with_extra,
    approach=PREPROCESS_PROFILE.get("approach"),
    sequence_length=seq_len,
    n_features=n_feat,
)

display(results_df.head(10))
print("Best params:", best_params)

## 7) Évaluation, robustesse et visualisations

In [ ]:
pred_test, metrics, classification_text_report = evaluate_test_set(
    final_model=final_model,
    X_test_imp=X_test_imp,
    y_test=y_test,
    model_profile=MODEL_PROFILE,
    target_profile=TARGET_PROFILE,
    show_plots=True,
)

print(pd.Series(metrics))
if classification_text_report is not None:
    print("\nClassification report:")
    print(classification_text_report)

by_subject = evaluate_by_subject(
    dataset_df=dataset_df,
    test_idx=test_idx,
    y_test=y_test,
    pred_test=pred_test,
    model_profile=MODEL_PROFILE,
)
display(by_subject.head(20))

imp_df = plot_feature_importance(final_model, feature_cols, top_n=15)
if imp_df is not None:
    display(imp_df.head(20))

noise_scores = evaluate_robustness(
    final_model=final_model,
    X_test_imp=X_test_imp,
    y_test=y_test,
    model_profile=MODEL_PROFILE,
    eval_profile={"robustness_noise_std": 0.01, "robustness_repeats": 5},
    seed=SEED,
    show_plot=True,
)
print("Score robustesse (moyenne) :", float(np.mean(noise_scores)))

plot_pca_if_classification(X_test_imp, y_test, MODEL_PROFILE, seed=SEED)

## 8) Fiche modèle

In [ ]:
export_visual_report(
    dataset_df=dataset_df,
    feature_cols=feature_cols,
    model_profile=MODEL_PROFILE,
    output_profile=OUTPUT_PROFILE,
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx,
    target_profile=TARGET_PROFILE,
    raw_df=raw_df,
    preprocess_profile=PREPROCESS_PROFILE,
    metrics=metrics,
    final_model=final_model,
    X_test_imp=X_test_imp,
    y_test=y_test,
)